In [2]:
from Bio import Phylo, SeqIO
from Bio.Seq import Seq

# загрузка
tree = Phylo.read("/home/ria/HSE/lab/model/knownpass/h1pdm.l.r.newick", "newick")
aln = {rec.id: str(rec.seq) for rec in SeqIO.parse("/home/ria/HSE/lab/model/knownpass/h1pdm.all.fa", "fasta")}

node_name = "Node103"
site = 202
site_idx = site - 1

node_seq = aln[node_name]
aa_node = str(Seq(node_seq).translate())

print("AA в Node103 на сайте 202:", aa_node[site_idx])

# найдём всех потомков Node103
node_clade = None
for clade in tree.find_clades():
    if clade.name == node_name:
        node_clade = clade
        break

print("\nПотомки с заменой на сайте 202:\n")

for clade in node_clade.find_clades():
    if clade.name and clade.name in aln:
        seq = aln[clade.name]
        aa = str(Seq(seq).translate())
        
        if aa[site_idx] != aa_node[site_idx]:
            dist = tree.distance(node_name, clade.name)
            print(clade.name,
                  "AA:", aa[site_idx],
                  "distance:", dist)


AA в Node103 на сайте 202: T

Потомки с заменой на сайте 202:

STRAIN1590379_2014 AA: S distance: 0.018121473
STRAIN586641_2015 AA: I distance: 0.02120395
STRAIN588341_2014 AA: A distance: 0.0149091373
STRAIN540119_2014 AA: N distance: 0.015696692
STRAIN717681_2015 AA: N distance: 0.025133595999999998
STRAIN842085_2016 AA: N distance: 0.025137699099999997
STRAIN1506823_2019 AA: I distance: 0.04798279580000002
STRAIN1351114_2018 AA: I distance: 0.04719711320000001
STRAIN1504255_2019 AA: N distance: 0.04725212790000001
STRAIN1315886_2018 AA: A distance: 0.04248349650000001
Node1250 AA: I distance: 0.0431161435
Node1251 AA: I distance: 0.043118234500000005
STRAIN1354800_2018 AA: I distance: 0.04389927550000001
Node1252 AA: I distance: 0.043120376200000005
STRAIN1384491_2018 AA: I distance: 0.045468976200000004
STRAIN1355904_2018 AA: I distance: 0.047051856200000006
STRAIN1433277_2019 AA: I distance: 0.0478347435
Node1369 AA: I distance: 0.039327537600000004
Node1370 AA: I distance: 0.0393

In [3]:
from Bio import Phylo

TREE_PATH = "/home/ria/HSE/lab/model/knownpass/h1pdm.l.r.newick"
START = "Node103"
TARGET = 0.0161
EPS = 1e-6  # допуск на сравнение float

tree = Phylo.read(TREE_PATH, "newick")

# найдём стартовый clade
start_clade = None
for cl in tree.find_clades():
    if cl.name == START:
        start_clade = cl
        break
if start_clade is None:
    raise ValueError(f"Не найден узел {START} в дереве")

# функция: обход вниз, накапливая расстояние
hits = []

def walk(clade, dist_so_far, path_names):
    # если уже прошли больше TARGET — дальше по этой ветке смысла нет
    if dist_so_far > TARGET + EPS:
        return

    # Если мы попали ровно в узел
    if abs(dist_so_far - TARGET) <= EPS:
        hits.append({
            "type": "node",
            "at": clade.name,
            "dist": dist_so_far,
            "path": " -> ".join(path_names + [clade.name if clade.name else "(unnamed)"])
        })
        # Даже если ровно в узел, можно продолжить (если есть нулевые ветви), но обычно не надо
        return

    # Идём по детям
    for ch in clade.clades:
        bl = ch.branch_length if ch.branch_length is not None else 0.0
        new_dist = dist_so_far + bl

        # Если точка TARGET лежит ВНУТРИ ребра clade -> ch
        if dist_so_far < TARGET - EPS and new_dist > TARGET + EPS:
            hits.append({
                "type": "edge",
                "from": clade.name,
                "to": ch.name,
                "dist_to_from": dist_so_far,
                "edge_len": bl,
                "offset_on_edge": TARGET - dist_so_far,  # сколько от начала ребра
                "target": TARGET,
                "path": " -> ".join(path_names + [clade.name if clade.name else "(unnamed)", ch.name if ch.name else "(unnamed)"])
            })
            # дальше по этому ребру идти не нужно, мы уже нашли точку
            continue

        # Если ещё не дошли — продолжаем
        walk(ch, new_dist, path_names + [clade.name if clade.name else "(unnamed)"])

walk(start_clade, 0.0, [])

print(f"Старт: {START}, target distance: {TARGET}")
print(f"Найдено попаданий (узлы или точки на ребрах): {len(hits)}")

# покажем первые 20
for h in hits[:20]:
    if h["type"] == "node":
        print(f"[NODE] at={h['at']} dist={h['dist']:.10f} path={h['path']}")
    else:
        print(
            f"[EDGE] {h['from']} -> {h['to']} | "
            f"edge_len={h['edge_len']:.10f} | "
            f"offset_on_edge={h['offset_on_edge']:.10f} | "
            f"path={h['path']}"
        )


Старт: Node103, target distance: 0.0161
Найдено попаданий (узлы или точки на ребрах): 149
[EDGE] Node134 -> Node135 | edge_len=0.0007528760 | offset_on_edge=0.0004559761 | path=Node103 -> Node104 -> Node105 -> Node106 -> Node118 -> Node119 -> Node120 -> Node121 -> Node122 -> Node123 -> Node125 -> Node126 -> Node127 -> Node128 -> Node129 -> Node132 -> Node133 -> Node134 -> Node135
[EDGE] Node134 -> STRAIN417468_2012 | edge_len=0.0048806400 | offset_on_edge=0.0004559761 | path=Node103 -> Node104 -> Node105 -> Node106 -> Node118 -> Node119 -> Node120 -> Node121 -> Node122 -> Node123 -> Node125 -> Node126 -> Node127 -> Node128 -> Node129 -> Node132 -> Node133 -> Node134 -> STRAIN417468_2012
[EDGE] Node141 -> STRAIN450243_2013 | edge_len=0.0023646500 | offset_on_edge=0.0019551241 | path=Node103 -> Node104 -> Node105 -> Node106 -> Node118 -> Node119 -> Node120 -> Node121 -> Node122 -> Node123 -> Node125 -> Node126 -> Node127 -> Node128 -> Node129 -> Node132 -> Node136 -> Node137 -> Node141 -

In [4]:
from Bio import Phylo, SeqIO
from Bio.Seq import Seq

TREE_PATH = "/home/ria/HSE/lab/model/knownpass/h1pdm.l.r.newick"
FASTA_PATH = "/home/ria/HSE/lab/model/knownpass/h1pdm.all.fa"

START = "Node103"
SITE = 202  # 1-based AA site

tree = Phylo.read(TREE_PATH, "newick")
aln = {rec.id: str(rec.seq) for rec in SeqIO.parse(FASTA_PATH, "fasta")}

def aa_at(name, site):
    """AA в узле name на позиции site (1-based)"""
    seq = aln.get(name)
    if seq is None:
        return None
    aa = str(Seq(seq).translate())
    i = site - 1
    if i < 0 or i >= len(aa):
        return None
    return aa[i]

# найти clade Node103
start = None
for cl in tree.find_clades():
    if cl.name == START:
        start = cl
        break
if start is None:
    raise ValueError(f"Не найден узел {START}")

start_aa = aa_at(START, SITE)
print("AA в Node103 на сайте", SITE, "=", start_aa)

hits = []  # сюда запишем все рёбра T->S

for parent in start.find_clades(order="preorder"):
    if parent.name is None:
        continue
    aap = aa_at(parent.name, SITE)
    if aap is None:
        continue

    for child in parent.clades:
        if child.name is None:
            continue
        aac = aa_at(child.name, SITE)
        if aac is None:
            continue

        if aap == "T" and aac == "S":
            # событие T->S на ребре parent->child
            edge_len = child.branch_length if child.branch_length is not None else 0.0
            dist_from_start_to_parent = tree.distance(START, parent.name)
            dist_from_start_to_child  = tree.distance(START, child.name)

            hits.append({
                "parent": parent.name,
                "child": child.name,
                "edge_len": edge_len,
                "dist_start_to_parent": dist_from_start_to_parent,
                "dist_start_to_child": dist_from_start_to_child
            })

print("\nНайдено рёбер T->S:", len(hits))
for h in hits[:20]:
    print(h)


AA в Node103 на сайте 202 = T

Найдено рёбер T->S: 2
{'parent': 'Node226', 'child': 'STRAIN1590379_2014', 'edge_len': 0.00397928, 'dist_start_to_parent': 0.014142192999999999, 'dist_start_to_child': 0.018121473}
{'parent': 'Node2047', 'child': 'STRAIN1349286_2018', 'edge_len': 0.00313962, 'dist_start_to_parent': 0.045588257199999996, 'dist_start_to_child': 0.0487278772}


Анализируем сайт: 202 
> site_data <- df_old %>% filter(site == popular_site)
> site_data %>% 
+   select(ancestor_node, grandanc_aa, anc_aa, der_aa, distance) %>% 
+   arrange(ancestor_node, distance)
 A tibble: 36 × 5
   ancestor_node grandanc_aa anc_aa der_aa distance
   <chr>         <chr>       <chr>  <chr>     <dbl>
 1 Node103       S           T      G        0.0103
 2 Node103       S           T      P        0.0126
 3 Node103       S           T      A        0.0137
 4 Node103       S           T      A        0.0145
 5 Node103       S           T      N        0.0149
 6 Node103       S           T      S        0.0161
 7 Node103       S           T      I        0.0192
 8 Node103       S           T      N        0.0212
 9 Node103       S           T      N        0.0232
10 Node103       S           T      N        0.0236
ℹ 26 more rows
ℹ Use `print(n = ...)` to see more rows
>  Для каждого ancestor_node посмотрим расстояния
> cat("\nПо ancestor_node:\n")

По ancestor_node:
> for(node in unique(site_data$ancestor_node)) {
+   cat("\n", node, ":\n", sep = "")
+   subset_data <- site_data %>% filter(ancestor_node == node)
+   print(subset_data %>% select(grandanc_aa, anc_aa, der_aa, distance))
+ }

Node103:
 A tibble: 31 × 4
   grandanc_aa anc_aa der_aa distance
   <chr>       <chr>  <chr>     <dbl>
 1 S           T      S        0.0161
 2 S           T      I        0.0192
 3 S           T      A        0.0137
 4 S           T      N        0.0149
 5 S           T      N        0.0236
 6 S           T      N        0.0232
 7 S           T      I        0.0464
 8 S           T      I        0.0456
 9 S           T      N        0.0449
10 S           T      A        0.0409
 ℹ 21 more rows
ℹ Use `print(n = ...)` to see more rows

Node1369:
 A tibble: 4 × 4
  grandanc_aa anc_aa der_aa distance
  <chr>       <chr>  <chr>     <dbl>
1 T           I      V       0.00591
2 T           I      T       0.00787
3 T           I      T       0.00827
4 T           I      V       0.00394

Node2227:
A tibble: 1 × 4
  grandanc_aa anc_aa der_aa distance
  <chr>       <chr>  <chr>     <dbl>
1 T           I      V       0.00396
> 

In [5]:
import pandas as pd
from Bio import Phylo, SeqIO
from Bio.Seq import Seq

TREE_PATH = "/home/ria/HSE/lab/model/knownpass/h1pdm.l.r.newick"
FASTA_PATH = "/home/ria/HSE/lab/model/knownpass/h1pdm.all.fa"
CSV_OLD = "/home/ria/HSE/lab/model/knownpass/h1pdm_distances.csv"

START = "Node103"
SITE = 202

tree = Phylo.read(TREE_PATH, "newick")
aln = {rec.id: str(rec.seq) for rec in SeqIO.parse(FASTA_PATH, "fasta")}

def aa_at(name, site):
    seq = aln.get(name)
    if seq is None:
        return None
    aa = str(Seq(seq).translate())
    i = site - 1
    return aa[i] if 0 <= i < len(aa) else None

# 1) найдём все рёбра T->S ниже Node103
start = next(cl for cl in tree.find_clades() if cl.name == START)

edges_TS = []
for parent in start.find_clades(order="preorder"):
    if not parent.name:
        continue
    aap = aa_at(parent.name, SITE)
    if aap is None:
        continue
    for child in parent.clades:
        if not child.name:
            continue
        aac = aa_at(child.name, SITE)
        if aac is None:
            continue
        if aap == "T" and aac == "S":
            edge_len = child.branch_length or 0.0
            d_parent = tree.distance(START, parent.name)
            d_child  = tree.distance(START, child.name)
            d_mid    = d_parent + 0.5 * edge_len
            edges_TS.append((parent.name, child.name, edge_len, d_parent, d_child, d_mid))

print("T->S edges:", len(edges_TS))
for e in edges_TS:
    print("parent", e[0], "child", e[1], "d_parent", e[3], "edge", e[2], "d_mid", e[5], "d_child", e[4])

# 2) возьмём distance из df_old и сравним с d_mid
df_old = pd.read_csv(CSV_OLD)

sub = df_old[
    (df_old["site"] == SITE) &
    (df_old["ancestor_node"] == START) &
    (df_old["anc_aa"] == "T") &
    (df_old["der_aa"] == "S")
].copy()

print("\nRows in df_old for Node103/site202 T->S:", len(sub))
print(sub[["grandanc_aa","anc_aa","der_aa","distance"]].head(20))

if len(sub) > 0 and len(edges_TS) > 0:
    # сравним каждую строку df_old со всеми d_mid
    for i, row in sub.iterrows():
        d = float(row["distance"])
        best = min(edges_TS, key=lambda e: abs(e[5] - d))
        print(
            f"\ndf_old distance={d:.10f} best_edge={best[0]}->{best[1]} "
            f"d_mid={best[5]:.10f} diff={abs(best[5]-d):.10e}"
        )


T->S edges: 2
parent Node226 child STRAIN1590379_2014 d_parent 0.014142192999999999 edge 0.00397928 d_mid 0.016131832999999998 d_child 0.018121473
parent Node2047 child STRAIN1349286_2018 d_parent 0.045588257199999996 edge 0.00313962 d_mid 0.047158067199999994 d_child 0.0487278772

Rows in df_old for Node103/site202 T->S: 2
    grandanc_aa anc_aa der_aa  distance
130           S      T      S  0.016132
145           S      T      S  0.047158

df_old distance=0.0161318330 best_edge=Node226->STRAIN1590379_2014 d_mid=0.0161318330 diff=3.4694469520e-18

df_old distance=0.0471580672 best_edge=Node2047->STRAIN1349286_2018 d_mid=0.0471580672 diff=6.9388939039e-18
